In [1]:
# Install Hugging Face transformers and datasets libraries
!pip install transformers datasets

# Install PyTorch (if not already installed)
!pip install torch

# Install pandas for data manipulation
!pip install pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 134.8/134.8 kB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.1/194.1 kB 10.3 MB/s eta 0:00:00
  Attempting uninstall: fsspec
    Found existing installation: fsspec 2024.10.0
    Uninstalling fsspec-2024.10.0:
      Successfully uninstalled fsspec-2024.10.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2024.10.0 requires fsspec==2024.10.0, but you have fsspec 2024.9.0 which is incompatible.


In [2]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, Trainer, TrainingArguments
from datasets import Dataset, DatasetDict


In [3]:
# Check for GPU
import torch
if torch.cuda.is_available():
    print("GPU is available!")
    device = torch.device("cuda")
else:
    print("Using CPU.")
    device = torch.device("cpu")


GPU is available!


In [4]:
from google.colab import files

# Upload sarcastic and non-sarcastic files
sarcastic_file = files.upload()  # Upload sarcastic.csv
non_sarcastic_file = files.upload()  # Upload non_sarcastic.csv


Saving Final_sarcastic_data.csv to Final_sarcastic_data.csv


Saving Final_non_sarcastic_data.csv to Final_non_sarcastic_data.csv


In [5]:
# Load CSV files into DataFrames
sarcastic_df = pd.read_csv("Final_sarcastic_data.csv")
non_sarcastic_df = pd.read_csv("Final_non_sarcastic_data.csv")

# Display the first few rows
print("Sarcastic Data:")
print(sarcastic_df.head())

print("\nNon-Sarcastic Data:")
print(non_sarcastic_df.head())


Sarcastic Data:
   id                                          sarcastic
0   1  Oh great, another meeting! My life is complete...
1   2  Wow, this traffic jam is *just* what I needed ...
2   3  Sure, giving me extra work at 5 PM is such a *...
3   4  Fantastic, my laptop decided to update in the ...
4   5  Your cooking is so good it belongs on a disast...

Non-Sarcastic Data:
   id                                    non_sarcastic
0   1                I dislike having another meeting.
1   2  This traffic jam is frustrating and unexpected.
2   3    Giving me extra work at 5 PM is inconvenient.
3   4  My laptop's update interrupted my presentation.
4   5                I don't like how the food tastes.


In [6]:
# 1. Merge the two DataFrames on the 'id' column
merged_df = pd.merge(sarcastic_df, non_sarcastic_df, on='id')

# 2. Convert text columns to strings
merged_df['sarcastic'] = merged_df['sarcastic'].astype(str)
merged_df['non_sarcastic'] = merged_df['non_sarcastic'].astype(str)

# 3. Display the first few rows of the merged DataFrame
print("Merged DataFrame:")
print(merged_df.head())


Merged DataFrame:
   id                                          sarcastic  \
0   1  Oh great, another meeting! My life is complete...   
1   2  Wow, this traffic jam is *just* what I needed ...   
2   3  Sure, giving me extra work at 5 PM is such a *...   
3   4  Fantastic, my laptop decided to update in the ...   
4   5  Your cooking is so good it belongs on a disast...   

                                     non_sarcastic  
0                I dislike having another meeting.  
1  This traffic jam is frustrating and unexpected.  
2    Giving me extra work at 5 PM is inconvenient.  
3  My laptop's update interrupted my presentation.  
4                I don't like how the food tastes.  


In [7]:
from sklearn.model_selection import train_test_split

# Split data into training and remaining sets
train_df, val_df = train_test_split(merged_df, test_size=0.2, random_state=42)

# Split remaining data into validation and test sets

print("\nData Splits:")
print(f"Training Set: {len(train_df)} samples")
print(f"Validation Set: {len(val_df)} samples")



Data Splits:
Training Set: 731 samples
Validation Set: 183 samples


In [8]:
from datasets import Dataset

# Convert pandas DataFrames to Hugging Face Dataset objects
train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

# Bundle them into a DatasetDict
dataset_dict = DatasetDict({
    'train': train_dataset,
    'validation': val_dataset,
})

print("\nDatasetDict Summary:")
print(dataset_dict)



DatasetDict Summary:
DatasetDict({
    train: Dataset({
        features: ['id', 'sarcastic', 'non_sarcastic', '__index_level_0__'],
        num_rows: 731
    })
    validation: Dataset({
        features: ['id', 'sarcastic', 'non_sarcastic', '__index_level_0__'],
        num_rows: 183
    })
})


In [9]:
!pip install transformers accelerate
!pip install transformers


In [10]:
from transformers import AutoTokenizer, AutoModelForCausalLM

# Load the tokenizer for DialoGPT
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")

# Load the DialoGPT model
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small")

# Move the model to the appropriate device (GPU or CPU)
model.to(device)

print("DialoGPT model and tokenizer loaded successfully!")


/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/641 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/351M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

DialoGPT model and tokenizer loaded successfully!


In [11]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from datasets import DatasetDict

# Load DialoGPT model and tokenizer
tokenizer = AutoTokenizer.from_pretrained("microsoft/DialoGPT-small")
model = AutoModelForCausalLM.from_pretrained("microsoft/DialoGPT-small")

# Check if the tokenizer has a padding token; add one if missing
if tokenizer.pad_token is None:
    print("Adding padding token...")
    tokenizer.pad_token = tokenizer.eos_token

# Resize the model's embedding layer if a new padding token is added
model.resize_token_embeddings(len(tokenizer))

# Move the model to the appropriate device (GPU or CPU)
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("Model and tokenizer are ready for fine-tuning!")

# Example dataset_dict: Ensure you have your dataset ready
# Assuming `dataset_dict` is already created as shown earlier
# dataset_dict = DatasetDict({'train': train_dataset, 'validation': val_dataset, 'test': test_dataset})

# Define preprocessing function
def preprocess_function(examples):
    # Combine sarcastic and non-sarcastic sentences into input-output pairs
    inputs = [
        f"Sarcastic: {sarcastic} [SEP] Non-Sarcastic: {non_sarcastic}"
        for sarcastic, non_sarcastic in zip(examples["sarcastic"], examples["non_sarcastic"])
    ]
    # Tokenize with padding and truncation
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"  # Ensures fixed-length tokenization
    )
    return model_inputs

# Apply tokenization to the dataset
tokenized_datasets = dataset_dict.map(preprocess_function, batched=True)

# Set PyTorch format
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask"])

print("Tokenization completed!")


Adding padding token...
Model and tokenizer are ready for fine-tuning!


Map:   0%|          | 0/731 [00:00<?, ? examples/s]

Map:   0%|          | 0/183 [00:00<?, ? examples/s]

Tokenization completed!


In [12]:
def preprocess_function(examples):
    # Combine sarcastic and non-sarcastic sentences into input-output pairs
    inputs = [
        f"Sarcastic: {sarcastic} [SEP] Non-Sarcastic: {non_sarcastic}"
        for sarcastic, non_sarcastic in zip(examples["sarcastic"], examples["non_sarcastic"])
    ]

    # Tokenize the inputs
    model_inputs = tokenizer(
        inputs,
        max_length=512,
        truncation=True,
        padding="max_length"
    )

    # Set labels (same as input_ids but pad tokens ignored)
    labels = model_inputs["input_ids"].copy()
    labels = [
        [label if label != tokenizer.pad_token_id else -100 for label in label_seq]
        for label_seq in labels
    ]

    model_inputs["labels"] = labels
    return model_inputs

# Apply tokenization and add labels
tokenized_datasets = dataset_dict.map(preprocess_function, batched=True)

# Set PyTorch format
tokenized_datasets.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

print("Dataset tokenized and labels added!")


Map:   0%|          | 0/731 [00:00<?, ? examples/s]

Map:   0%|          | 0/183 [00:00<?, ? examples/s]

Dataset tokenized and labels added!


In [16]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./dialoGPT-results",     # Directory to save checkpoints
    evaluation_strategy="epoch",        # Evaluate after each epoch
    learning_rate=5e-5,                 # Learning rate
    per_device_train_batch_size=4,      # Batch size for training
    per_device_eval_batch_size=4,       # Batch size for evaluation
    num_train_epochs=7,                 # Number of epochs
    save_total_limit=2,                 # Save only the last 2 checkpoints
    weight_decay=0.01,                  # Weight decay
    logging_dir="./logs",               # Directory for logs
    logging_steps=100,
    report_to="none",# Log every 100 steps
    fp16=True if torch.cuda.is_available() else False
    # Mixed precision if possible
)


/usr/local/lib/python3.10/dist-packages/transformers/training_args.py:1568: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [17]:
from transformers import Trainer

trainer = Trainer(
    model=model,                               # The DialoGPT model
    args=training_args,                        # Training arguments
    train_dataset=tokenized_datasets["train"], # Training data
    eval_dataset=tokenized_datasets["validation"],  # Validation data
    tokenizer=tokenizer                        # Tokenizer
)


<ipython-input-17-0765555e242a>:3: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


In [18]:
trainer.train()


Epoch,Training Loss,Validation Loss
1,1.315200,1.625377
2,1.170500,1.571082
3,1.051400,1.583576
4,0.932500,1.626797
5,0.821600,1.661977
6,0.756100,1.690593
7,0.709100,1.708825


TrainOutput(global_step=1281, training_loss=0.9486183714438564, metrics={'train_runtime': 414.1024, 'train_samples_per_second': 12.357, 'train_steps_per_second': 3.093, 'total_flos': 1337031327744000.0, 'train_loss': 0.9486183714438564, 'epoch': 7.0})

In [19]:
!pip install rouge_score
!pip install evaluate


  Preparing metadata (setup.py) ... done
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24935 sha256=04f50554b72cb6576db77721bd0db9081edad4b4aa6e5b87e35cf9fb28b3b68f
  Stored in directory: /root/.cache/pip/wheels/5f/dd/89/461065a73be61a532ff8599a28e9beef17985c9e9c31e541b4
Successfully built rouge_score
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.0/84.0 kB 8.0 MB/s eta 0:00:00


In [21]:
# Save the model and tokenizer
model.save_pretrained("./sarcastic_model")
tokenizer.save_pretrained("./sarcastic_model")




('./sarcastic_model/tokenizer_config.json',
 './sarcastic_model/special_tokens_map.json',
 './sarcastic_model/vocab.json',
 './sarcastic_model/merges.txt',
 './sarcastic_model/added_tokens.json',
 './sarcastic_model/tokenizer.json')

In [43]:
# Install necessary libraries
!pip install nltk rouge-score pandas

from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import pandas as pd

# Load the predictions and reference data
input_data = pd.read_csv("output_predictions.csv")

# Initialize BLEU scorer
smoothie = SmoothingFunction().method4
bleu_scores = []

# Initialize ROUGE scorer
scorer = rouge_scorer.RougeScorer(['rouge1', 'rouge2', 'rougeL'], use_stemmer=True)
rouge_scores = {"rouge1": [], "rouge2": [], "rougeL": []}

# Calculate BLEU and ROUGE scores for each prediction
for reference, prediction in zip(input_data["text"], input_data["generated_sarcastic_text"]):
    # BLEU Score
    reference_tokens = reference.split()  # Tokenize reference
    prediction_tokens = prediction.split()  # Tokenize prediction
    bleu_score = sentence_bleu([reference_tokens], prediction_tokens, smoothing_function=smoothie)
    bleu_scores.append(bleu_score)

    # ROUGE Scores
    rouge = scorer.score(reference, prediction)
    rouge_scores["rouge1"].append(rouge["rouge1"].fmeasure)
    rouge_scores["rouge2"].append(rouge["rouge2"].fmeasure)
    rouge_scores["rougeL"].append(rouge["rougeL"].fmeasure)

# Add BLEU and ROUGE scores to the DataFrame
input_data["BLEU_score"] = bleu_scores
input_data["ROUGE-1"] = rouge_scores["rouge1"]
input_data["ROUGE-2"] = rouge_scores["rouge2"]
input_data["ROUGE-L"] = rouge_scores["rougeL"]

# Save results to a new CSV file
input_data.to_csv("output_with_metrics.csv", index=False)

# Print average scores
print(f"Average BLEU score: {sum(bleu_scores) / len(bleu_scores):.4f}")
print(f"Average ROUGE-1 score: {sum(rouge_scores['rouge1']) / len(rouge_scores['rouge1']):.4f}")
print(f"Average ROUGE-2 score: {sum(rouge_scores['rouge2']) / len(rouge_scores['rouge2']):.4f}")
print(f"Average ROUGE-L score: {sum(rouge_scores['rougeL']) / len(rouge_scores['rougeL']):.4f}")


Average BLEU score: 0.6313
Average ROUGE-1 score: 0.7254
Average ROUGE-2 score: 0.7171
Average ROUGE-L score: 0.7254


In [42]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch

# Load the fine-tuned model and tokenizer
model_name = "./sarcastic_model"  # Adjust path to your model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# Define the function to generate non-sarcastic text
def generate_non_sarcastic(sarcastic_text):
    # Prepare input
    input_text = f"Sarcastic: {sarcastic_text} Non-Sarcastic:"
    inputs = tokenizer(input_text, return_tensors="pt").to(device)

    # Generate output
    outputs = model.generate(
        **inputs,
        max_length=100,
        num_return_sequences=1,
        pad_token_id=tokenizer.eos_token_id
    )

    # Decode the generated text
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result.split("[SEP] Non-Sarcastic:")[-1].strip()

# Load the sarcastic dataset
input_file = "Sarcastic_IAC_data.csv"  # Adjust path if needed
output_file = "non_sarcastic_output.csv"  # Output file
data = pd.read_csv(input_file)

# Ensure the dataset has the correct column
if "text" not in data.columns:
    raise ValueError("The input file must have a 'sarcastic' column.")

# Generate non-sarcastic text for each row
non_sarcastic_texts = []
for sarcastic_text in data["text"]:
    try:
        non_sarcastic_text = generate_non_sarcastic(sarcastic_text)
        non_sarcastic_texts.append(non_sarcastic_text)
    except Exception as e:
        print(f"Error processing: {sarcastic_text}")
        non_sarcastic_texts.append("")

# Add results to the DataFrame and save to file
data["non_sarcastic"] = non_sarcastic_texts
data.to_csv(output_file, index=False)
print(f"Non-sarcastic text saved to {output_file}")


Error processing: I just bet you thought you were smart just now? I'll go you several... 
How about:
1. Murder
2. Stealing
3. Adultery
4. Rape
Almost every culture since the dawn of civilization has put to death those who have trespassed those basics. But I guess you didn't get that on the Recovery Channel huh? Your idiocy is profound only as your ignorance.
Error processing: Wow, what a group of rude, self important, pompous, pseudo intellectuals who will insult and personally attack a person for one grammatical error. I meant to say the UNIVERSE has aged from 4 to 14 billion years old since I was in school but rather than deduce that you call me dumb and ignorant ? With all of your so called intelligence can you prove the Universe is now precisely 14 billion years old ? Can you prove Neanderthal Man was a human descendant ? because DNA tests dont prove that out and yet PHDs and Masters Degreed scientists insist they are. That's why the bible say's that the wisdom of this World is foo